In [ ]:

import os, sys, subprocess, warnings, shutil, re
warnings.filterwarnings("ignore")

def install(pkg_string):
    pkgs = pkg_string.split()
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

install("ultralytics opencv-python pandas tqdm pillow facenet-pytorch")
install("git+https://github.com/KaiyangZhou/deep-person-reid.git")

import cv2
import torch
import numpy as np
import pandas as pd
import zipfile
from tqdm import tqdm
from dataclasses import dataclass, field
from ultralytics import YOLO
from torchreid.utils import FeatureExtractor
from facenet_pytorch import MTCNN, InceptionResnetV1

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# =========================
# CONFIG: EDIT THESE PATHS
# =========================

@dataclass
class Config:
    INPUT_VIDEO: str = "/content/drive/MyDrive/my_word/WIN_20260218_14_25_12_Pro.mp4"
    OSNET_CHECKPOINT: str = "/content/drive/MyDrive/best_model.pth"

    OUTPUT_ROOT_BASE: str = "/content/reid_botsort_output"
    RESET_OUTPUT_EACH_RUN: bool = True

    YOLO_MODEL: str = "yolov8m.pt"
    TRACKER_CFG: str = "botsort.yaml"
    OSNET_MODEL_NAME: str = "osnet_ain_x1_0"

    PERSON_CLASS_ID: int = 0
    DET_CONF: float = 0.35
    MATCH_THRESHOLD: float = 0.73
    MATCH_MARGIN: float = 0.05

    MIN_CROP_WIDTH: int = 35
    MIN_CROP_HEIGHT: int = 90
    HUMAN_ASPECT_MIN: float = 1.20
    HUMAN_ASPECT_MAX: float = 4.80

    FRAME_BOUNDARY_MARGIN: int = 6

    # --- FIX: stricter blur threshold, was 80.0 ---
    BLUR_THRESHOLD: float = 130.0

    MIN_BRIGHTNESS: float = 35.0
    MAX_BRIGHTNESS: float = 220.0
    MIN_CONTRAST: float = 22.0
    MAX_NOISE_SCORE: float = 35.0

    # --- NEW knobs for motion blur + thin sliver detection ---
    DIRECTIONAL_BLUR_MIN_RATIO: float = 0.35   # below this => motion smear
    MIN_VISIBLE_BODY_RATIO: float = 0.55       # below this => thin sliver / mostly occluded

    # --- Occlusion: soft thresholds (flag + quality penalty, crop still USABLE) ---
    IOU_OCCLUSION_THRESHOLD: float = 0.25
    SMALLER_BOX_OVERLAP_THRESHOLD: float = 0.45

    SEVERE_IOU_OCCLUSION_THRESHOLD: float = 0.55
    SEVERE_SMALLER_BOX_OVERLAP_THRESHOLD: float = 0.75

    # --- FIX: require more good candidates + higher bar before committing ---
    MIN_GOOD_CANDIDATES_BEFORE_REID: int = 8
    READY_QUALITY_SCORE: float = 0.68
    MAX_WAIT_FRAMES_FOR_TRACK: int = 90

    ALLOW_FALLBACK_CROP: bool = True
    # --- FIX: fallback floor raised hard, was 0.12 (basically accept-anything) ---
    FALLBACK_MIN_QUALITY: float = 0.45
    PROCESS_LOW_QUALITY_AT_END: bool = True

    DRAW_PENDING_TRACKS: bool = True
    SAVE_GALLERY_FEATURES_NPY: bool = True

    TRACK_HIGH_THRESH: float = 0.55      # was 0.5 (default) -> fewer weak/ghost tracks
    TRACK_LOW_THRESH: float = 0.10
    NEW_TRACK_THRESH: float = 0.70       # raise bar before spawning a brand-new ID
    TRACK_BUFFER: int = 90               # was 30 (default) -> keep a lost track "alive"
                                          # much longer through occlusion before giving up
    MATCH_THRESH_TRACKER: float = 0.80
    PROXIMITY_THRESH: float = 0.55
    APPEARANCE_THRESH: float = 0.30
    TRACKER_WITH_REID: bool = True       # use BoT-SORT's own appearance ReID to
                                          # re-associate a track across occlusion,
                                          # independent of the final OSNet stage

    # =========================
    # NEW: FACE (MTCNN detect + FaceNet embedding) FUSION
    # =========================
    # The final identity feature is a weighted fusion of:
    #     final_feature = (1 - FACE_WEIGHT) * body_feature(OSNet)
    #                   +  FACE_WEIGHT      * face_feature(FaceNet)
    # FACE_WEIGHT is kept VERY LOW right now on purpose, so the body/OSNet
    # feature still dominates and your current Re-ID result stays basically
    # the same as before. Later, just raise FACE_WEIGHT (e.g. 0.3, 0.5...)
    # to see how much the face embedding changes the result -- no other
    # code needs to change.
    USE_FACE_FEATURE: bool = True
    FACE_WEIGHT: float = 0           # <-- low on purpose, raise later to test
    FACE_MIN_PROB: float = 0.90          # MTCNN detection confidence floor
    FACE_MIN_SIZE: int = 20              # MTCNN min_face_size in pixels
    FACE_IMG_SIZE: int = 160             # FaceNet/InceptionResnetV1 input size

CFG = Config()

if not os.path.exists(CFG.INPUT_VIDEO):
    raise FileNotFoundError(f"Input video not found: {CFG.INPUT_VIDEO}")

if not os.path.exists(CFG.OSNET_CHECKPOINT):
    raise FileNotFoundError(f"OSNet checkpoint not found: {CFG.OSNET_CHECKPOINT}")

def safe_video_stem(path):
    stem = os.path.splitext(os.path.basename(path))[0] or "video"
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", stem).strip("_") or "video"

def prepare_output_paths(cfg):
    video_stem = safe_video_stem(cfg.INPUT_VIDEO)
    output_root = os.path.join(cfg.OUTPUT_ROOT_BASE, f"current_video_{video_stem}")

    protected = {
        os.path.abspath(os.sep),
        os.path.abspath("/content"),
        os.path.abspath("/content/drive"),
        os.path.abspath(cfg.OUTPUT_ROOT_BASE),
    }
    abs_output_root = os.path.abspath(output_root)

    if cfg.RESET_OUTPUT_EACH_RUN and os.path.isdir(abs_output_root):
        if abs_output_root in protected:
            raise RuntimeError(f"Refusing to clean protected output path: {abs_output_root}")
        shutil.rmtree(abs_output_root)

    crops_dir = os.path.join(abs_output_root, "selected_track_crops")
    gallery_dir = os.path.join(abs_output_root, "identity_gallery")
    os.makedirs(crops_dir, exist_ok=True)
    os.makedirs(gallery_dir, exist_ok=True)

    return {
        "OUTPUT_ROOT": abs_output_root,
        "CROPS_DIR": crops_dir,
        "GALLERY_DIR": gallery_dir,
        "OUTPUT_VIDEO": os.path.join(abs_output_root, "final_annotated_video.mp4"),
        "DECISIONS_CSV": os.path.join(abs_output_root, "identity_decisions.csv"),
        "RANKINGS_CSV": os.path.join(abs_output_root, "ranking_scores.csv"),
        "MAPPING_CSV": os.path.join(abs_output_root, "tracker_to_final_identity.csv"),
        "GALLERY_CSV": os.path.join(abs_output_root, "identity_gallery.csv"),
        "FEATURES_NPY": os.path.join(abs_output_root, "gallery_features.npy"),
        # --- NEW: summary CSV (all counts + ids together) and crops zip ---
        "SUMMARY_CSV": os.path.join(abs_output_root, "summary_counts_and_ids.csv"),
        "CROPS_ZIP": os.path.join(abs_output_root, "person_crops.zip"),
    }

PATHS = prepare_output_paths(CFG)
CROPS_DIR = PATHS["CROPS_DIR"]
GALLERY_DIR = PATHS["GALLERY_DIR"]
OUTPUT_VIDEO = PATHS["OUTPUT_VIDEO"]
DECISIONS_CSV = PATHS["DECISIONS_CSV"]
RANKINGS_CSV = PATHS["RANKINGS_CSV"]
MAPPING_CSV = PATHS["MAPPING_CSV"]
GALLERY_CSV = PATHS["GALLERY_CSV"]
SUMMARY_CSV = PATHS["SUMMARY_CSV"]
CROPS_ZIP = PATHS["CROPS_ZIP"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("Current-video output folder:", PATHS["OUTPUT_ROOT"])

# =========================
# FIX: custom BoT-SORT tracker config (stronger occlusion survival)
# Stock botsort.yaml has a short track_buffer and no appearance ReID inside
# the tracker itself, so a person occluded for more than ~1 second gets
# dropped and reappears as a brand-new tracker_id -> inflates/distorts the
# Total Unique ID count downstream. This custom yaml fixes that at the
# tracking layer, before anything reaches OSNet.
# =========================
CUSTOM_TRACKER_YAML = os.path.join(PATHS["OUTPUT_ROOT"], "botsort_custom.yaml")
with open(CUSTOM_TRACKER_YAML, "w") as f:
    f.write(f"""tracker_type: botsort
track_high_thresh: {CFG.TRACK_HIGH_THRESH}
track_low_thresh: {CFG.TRACK_LOW_THRESH}
new_track_thresh: {CFG.NEW_TRACK_THRESH}
track_buffer: {CFG.TRACK_BUFFER}
match_thresh: {CFG.MATCH_THRESH_TRACKER}
fuse_score: True
gmc_method: sparseOptFlow
proximity_thresh: {CFG.PROXIMITY_THRESH}
appearance_thresh: {CFG.APPEARANCE_THRESH}
with_reid: {CFG.TRACKER_WITH_REID}
model: auto
""")
CFG.TRACKER_CFG = CUSTOM_TRACKER_YAML
print("Using custom tracker config:", CFG.TRACKER_CFG)

# =========================
# UTILITIES
# =========================

def clamp_bbox(box, w, h):
    x1, y1, x2, y2 = map(int, box)
    return max(0, x1), max(0, y1), min(w - 1, x2), min(h - 1, y2)

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)

    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter + 1e-9

    return inter / union, inter / (min(area_a, area_b) + 1e-9)

def l2_normalize(x):
    x = np.asarray(x, dtype=np.float32)
    return x / (np.linalg.norm(x) + 1e-12)

def identity_name(identity_num):
    return f"ID-{identity_num:03d}"

def rejection_reasons(flags):
    return ",".join([k for k, v in (flags or {}).items() if v]) or "none"

# --- NEW: motion-blur detector via directional gradient balance ---
def directional_blur_score(gray):
    """
    Sharp / non-motion-blurred crops have roughly balanced gradient energy in
    x and y directions. Motion-smeared crops have strong gradient in ONE direction
    and weak in the perpendicular one. Returns ratio in [0, 1]; LOW = likely motion blur.
    """
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    energy_x = float(np.mean(np.abs(gx)))
    energy_y = float(np.mean(np.abs(gy)))
    lo, hi = min(energy_x, energy_y), max(energy_x, energy_y)
    return lo / (hi + 1e-6)

# --- NEW: thin-sliver / heavy-occlusion detector via edge column coverage ---
def visible_body_ratio(gray):
    """
    Measures what fraction of the crop's width actually contains edge/body content.
    A thin vertical sliver (e.g. person mostly hidden behind a wall/doorway, only an
    arm or leg visible) will have edges concentrated in a small fraction of columns.
    Returns ratio in [0, 1]; LOW = thin sliver / mostly empty crop.
    """
    edges = cv2.Canny(gray, 50, 150)
    col_density = edges.sum(axis=0)
    active_cols = int(np.sum(col_density > 0))
    return active_cols / max(gray.shape[1], 1)

# =========================
# CROP QUALITY
# =========================

class CropQualityEvaluator:
    def __init__(self, cfg):
        self.cfg = cfg

    def evaluate(self, frame, bbox, all_bboxes, det_conf):
        fh, fw = frame.shape[:2]
        x1, y1, x2, y2 = clamp_bbox(bbox, fw, fh)
        bw, bh = x2 - x1, y2 - y1

        flags = {
            "is_occluded": False,
            "is_severely_occluded": False,
            "is_blurry": False,
            "is_motion_blurred": False,   # NEW
            "is_thin_sliver": False,      # NEW
            "is_full_body": True,
            "touches_boundary": False,
            "bad_size": False,
            "bad_aspect": False,
            "bad_brightness": False,
            "low_contrast": False,
            "is_noisy": False,
            "low_confidence": False,
        }

        if det_conf < self.cfg.DET_CONF:
            flags["low_confidence"] = True

        margin = self.cfg.FRAME_BOUNDARY_MARGIN
        if x1 <= margin or y1 <= margin or x2 >= fw - margin or y2 >= fh - margin:
            flags["touches_boundary"] = True
            flags["is_full_body"] = False

        if bw < self.cfg.MIN_CROP_WIDTH or bh < self.cfg.MIN_CROP_HEIGHT:
            flags["bad_size"] = True
            flags["is_full_body"] = False

        aspect = bh / max(bw, 1)
        if aspect < self.cfg.HUMAN_ASPECT_MIN or aspect > self.cfg.HUMAN_ASPECT_MAX:
            flags["bad_aspect"] = True
            flags["is_full_body"] = False

        for other in all_bboxes:
            if tuple(map(int, other)) == tuple(map(int, bbox)):
                continue
            box_iou, smaller_overlap = iou_xyxy((x1, y1, x2, y2), other)
            if box_iou >= self.cfg.IOU_OCCLUSION_THRESHOLD or smaller_overlap >= self.cfg.SMALLER_BOX_OVERLAP_THRESHOLD:
                flags["is_occluded"] = True
            # --- FIX: only PAST the severe line do we hard-reject the crop.
            # Below that, is_occluded stays True (still penalized in the score
            # below) but the crop remains usable, instead of every partial
            # occlusion in a crowded scene starving the track of any valid crop.
            if box_iou >= self.cfg.SEVERE_IOU_OCCLUSION_THRESHOLD or smaller_overlap >= self.cfg.SEVERE_SMALLER_BOX_OVERLAP_THRESHOLD:
                flags["is_severely_occluded"] = True
                break

        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            return False, -1.0, flags, None

        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

        blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
        brightness = float(np.mean(gray))
        contrast = float(np.std(gray))

        denoised = cv2.medianBlur(gray, 3)
        noise_score = float(np.std(gray.astype(np.float32) - denoised.astype(np.float32)))

        # --- NEW checks ---
        dir_blur_ratio = directional_blur_score(gray)
        body_ratio = visible_body_ratio(gray)

        if blur_score < self.cfg.BLUR_THRESHOLD:
            flags["is_blurry"] = True
        if dir_blur_ratio < self.cfg.DIRECTIONAL_BLUR_MIN_RATIO:
            flags["is_motion_blurred"] = True
        if body_ratio < self.cfg.MIN_VISIBLE_BODY_RATIO:
            flags["is_thin_sliver"] = True
            flags["is_full_body"] = False
        if brightness < self.cfg.MIN_BRIGHTNESS or brightness > self.cfg.MAX_BRIGHTNESS:
            flags["bad_brightness"] = True
        if contrast < self.cfg.MIN_CONTRAST:
            flags["low_contrast"] = True
        if noise_score > self.cfg.MAX_NOISE_SCORE and blur_score < self.cfg.BLUR_THRESHOLD * 2:
            flags["is_noisy"] = True

        invalid = any([
            flags["low_confidence"],
            flags["touches_boundary"],
            flags["bad_size"],
            flags["bad_aspect"],
            flags["is_blurry"],
            flags["is_motion_blurred"],
            flags["is_thin_sliver"],
            flags["bad_brightness"],
            flags["low_contrast"],
            flags["is_noisy"],
            flags["is_severely_occluded"],   # FIX: was is_occluded (too strict)
            not flags["is_full_body"],
        ])

        blur_norm = min(blur_score / 500.0, 1.0)
        conf_norm = min(max(det_conf, 0), 1)
        brightness_norm = 1.0 - abs(brightness - 128.0) / 128.0
        contrast_norm = min(contrast / 70.0, 1.0)
        size_norm = min((bh / max(fh, 1)) / 0.55, 1.0)

        quality = (
            0.30 * blur_norm +
            0.25 * conf_norm +
            0.20 * contrast_norm +
            0.15 * brightness_norm +
            0.10 * size_norm
        )

        # --- FIX: occlusion penalty now scales with severity instead of one
        # flat hard cut. Soft (partial) occlusion costs less than before so a
        # decent partially-occluded crop can still win against a worse clean
        # one; severe occlusion still costs heavily and is also hard-rejected
        # above via `invalid`. ---
        if flags["is_severely_occluded"]:
            quality -= 0.45
        elif flags["is_occluded"]:
            quality -= 0.20
        if flags["touches_boundary"]:
            quality -= 0.25
        if not flags["is_full_body"]:
            quality -= 0.30
        # --- NEW penalties ---
        if flags["is_motion_blurred"]:
            quality -= 0.30
        if flags["is_thin_sliver"]:
            quality -= 0.30

        quality = float(max(0.0, min(1.0, quality)))
        return not invalid, quality, flags, crop.copy()

# =========================
# BEST CROP MANAGER
# =========================

@dataclass
class TrackCropRecord:
    tracker_id: int
    first_seen_frame: int
    last_seen_frame: int
    candidate_count: int = 0
    valid_candidate_count: int = 0
    best_crop: np.ndarray = None
    best_quality_score: float = -1.0
    best_frame_number: int = -1
    best_bbox: tuple = None
    best_flags: dict = field(default_factory=dict)
    best_crop_was_valid: bool = False
    processed: bool = False
    final_identity_id: str = None
    match_score: float = None
    decision_type: str = None
    crop_path: str = None
    has_face: bool = False

class BestCropManager:
    def __init__(self, cfg):
        self.cfg = cfg
        self.records = {}

    def observe_track(self, tracker_id, frame_number):
        if tracker_id not in self.records:
            self.records[tracker_id] = TrackCropRecord(
                tracker_id=tracker_id,
                first_seen_frame=frame_number,
                last_seen_frame=frame_number
            )
        self.records[tracker_id].last_seen_frame = frame_number

    def update_if_better(self, tracker_id, frame_number, bbox, crop, quality_score, flags, is_valid):
        rec = self.records[tracker_id]
        rec.candidate_count += 1
        if is_valid:
            rec.valid_candidate_count += 1

        # --- FIX: also treat motion-blurred / thin-sliver crops as "worse than a clean
        # crop of similar score" by folding them into the is_valid comparison. A crop
        # that is technically "valid" (passed strict checks) but flagged motion-blurred
        # should not be preferred over a clean fallback once we already raised the floor.
        is_clean = is_valid and not flags.get("is_motion_blurred", False) and not flags.get("is_thin_sliver", False)
        rec_is_clean = rec.best_crop_was_valid and not rec.best_flags.get("is_motion_blurred", False) and not rec.best_flags.get("is_thin_sliver", False)

        should_replace = False
        if rec.best_crop is None:
            should_replace = True
        elif is_clean and not rec_is_clean:
            should_replace = True
        elif is_clean == rec_is_clean and is_valid and not rec.best_crop_was_valid:
            should_replace = True
        elif is_clean == rec_is_clean and is_valid == rec.best_crop_was_valid and quality_score > rec.best_quality_score:
            should_replace = True

        if should_replace:
            rec.best_crop = crop
            rec.best_quality_score = quality_score
            rec.best_frame_number = frame_number
            rec.best_bbox = tuple(map(int, bbox))
            rec.best_flags = dict(flags)
            rec.best_crop_was_valid = bool(is_valid)

    def is_ready(self, tracker_id, current_frame):
        rec = self.records[tracker_id]
        if rec.processed or rec.best_crop is None:
            return False

        has_enough_good_valid_crops = (
            rec.valid_candidate_count >= self.cfg.MIN_GOOD_CANDIDATES_BEFORE_REID
            and rec.best_crop_was_valid
            and rec.best_quality_score >= self.cfg.READY_QUALITY_SCORE
        )

        waited_for_good_valid_crop = (
            current_frame - rec.first_seen_frame >= self.cfg.MAX_WAIT_FRAMES_FOR_TRACK
            and rec.best_crop_was_valid
            and rec.best_quality_score >= self.cfg.READY_QUALITY_SCORE
        )

        waited_for_fallback_crop = (
            self.cfg.ALLOW_FALLBACK_CROP
            and current_frame - rec.first_seen_frame >= self.cfg.MAX_WAIT_FRAMES_FOR_TRACK
            and rec.best_quality_score >= self.cfg.FALLBACK_MIN_QUALITY
        )

        return has_enough_good_valid_crops or waited_for_good_valid_crop or waited_for_fallback_crop

    def unprocessed_with_crop(self):
        return [
            rec for rec in self.records.values()
            if not rec.processed and rec.best_crop is not None
        ]

# =========================
# OSNET FEATURE EXTRACTOR
# =========================

class OSNetFeatureExtractor:
    def __init__(self, cfg, device):
        self.extractor = FeatureExtractor(
            model_name=cfg.OSNET_MODEL_NAME,
            model_path=cfg.OSNET_CHECKPOINT,
            device=device
        )

    def extract(self, image_path):
        with torch.no_grad():
            feat = self.extractor([image_path])
            if isinstance(feat, torch.Tensor):
                feat = feat.detach().cpu().numpy()
            feat = np.asarray(feat)[0]
        return l2_normalize(feat)

# =========================
# NEW: FACE FEATURE EXTRACTOR (MTCNN detect + FaceNet embed)
# =========================
# Runs ONLY on the already-selected best body crop for each track (not on every
# frame), so it adds very little extra compute. If MTCNN finds no face in the
# crop (person facing away, too small, occluded, etc.) this simply returns
# None and the pipeline falls back to pure OSNet body feature for that track,
# exactly like before this change.

class FaceFeatureExtractor:
    def __init__(self, cfg, device):
        self.cfg = cfg
        self.mtcnn = MTCNN(
            min_face_size=cfg.FACE_MIN_SIZE,
            keep_all=False,          # only the most confident face in the crop
            select_largest=True,
            post_process=True,
            device=device
        )
        self.facenet = InceptionResnetV1(pretrained="vggface2").eval().to(device)
        self.device = device

    def extract(self, bgr_crop):
        """
        bgr_crop: numpy BGR person crop (same crop used for OSNet).
        Returns: l2-normalized 512-d face embedding (np.ndarray), or None if
        no face was detected with enough confidence.
        """
        if bgr_crop is None or bgr_crop.size == 0:
            return None

        rgb = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2RGB)

        try:
            face_tensor, prob = self.mtcnn(rgb, return_prob=True)
        except Exception:
            return None

        if face_tensor is None or prob is None:
            return None
        if isinstance(prob, (list, tuple, np.ndarray)):
            prob = prob[0] if len(prob) else None
        if prob is None or prob < self.cfg.FACE_MIN_PROB:
            return None

        with torch.no_grad():
            face_tensor = face_tensor.unsqueeze(0).to(self.device)
            embedding = self.facenet(face_tensor)
            embedding = embedding.detach().cpu().numpy()[0]

        return l2_normalize(embedding)

# =========================
# IDENTITY GALLERY + MATCHER
# =========================

class IdentityGallery:
    def __init__(self):
        self.next_id = 1
        self.items = {}

    def is_empty(self):
        return len(self.items) == 0

    def create_identity(self, feature, crop_path, frame_number, score=None, face_feature=None):
        identity_id = identity_name(self.next_id)
        self.next_id += 1

        self.items[identity_id] = {
            "features": [feature],
            "face_features": [face_feature] if face_feature is not None else [],
            "representative_image": crop_path,
            "first_seen_frame": frame_number,
            "last_seen_frame": frame_number,
            "last_score": score,
        }
        return identity_id

    def update_identity(self, identity_id, feature, crop_path, frame_number, score, face_feature=None):
        self.items[identity_id]["features"].append(feature)
        if face_feature is not None:
            self.items[identity_id]["face_features"].append(face_feature)
        self.items[identity_id]["last_seen_frame"] = frame_number
        self.items[identity_id]["last_score"] = score

    def rows(self):
        data = []
        for identity_id, item in self.items.items():
            data.append({
                "identity_id": identity_id,
                "num_features": len(item["features"]),
                "representative_image": item["representative_image"],
                "first_seen_frame": item["first_seen_frame"],
                "last_seen_frame": item["last_seen_frame"],
                "last_score": item["last_score"],
            })
        return data

class ReIDMatcher:
    def __init__(self, gallery):
        self.gallery = gallery

    def rank(self, query_feature, query_face_feature=None, face_weight=0.0):
        """
        Fused similarity score per identity:
            final_score = (1 - face_weight) * body_similarity
                        +  face_weight      * face_similarity
        face_similarity only contributes when BOTH the query crop AND at
        least one stored gallery sample for that identity have a usable
        face embedding -- otherwise that identity's score is pure body
        similarity, so identities/tracks with no visible face are completely
        unaffected by this change.
        """
        ranking = []
        for identity_id, item in self.gallery.items.items():
            body_score = max(float(np.dot(query_feature, f)) for f in item["features"])

            face_score = None
            gallery_faces = item.get("face_features") or []
            if query_face_feature is not None and gallery_faces:
                face_score = max(float(np.dot(query_face_feature, f)) for f in gallery_faces)

            if face_score is not None and face_weight > 0.0:
                fused_score = (1.0 - face_weight) * body_score + face_weight * face_score
            else:
                fused_score = body_score

            ranking.append((identity_id, fused_score))
        ranking.sort(key=lambda x: x[1], reverse=True)
        return ranking

    def decide(self, query_feature, threshold, margin, query_face_feature=None, face_weight=0.0):
        if self.gallery.is_empty():
            return "new", None, None, None, None, []

        ranking = self.rank(query_feature, query_face_feature, face_weight)

        best_id, best_score = ranking[0]
        if len(ranking) > 1:
            second_id, second_score = ranking[1]
        else:
            second_id, second_score = None, -1.0

        if best_score >= threshold and (best_score - second_score) >= margin:
            return "old", best_id, best_score, second_id, second_score, ranking

        return "new", best_id, best_score, second_id, second_score, ranking

# =========================
# LOGGING
# =========================

class DecisionLogger:
    def __init__(self):
        self.decisions = []
        self.rankings = []
        self.mappings = []

    def log_decision(self, rec, decision_type, best_match_id, best_score, second_match_id, second_score, cfg):
        x1, y1, x2, y2 = rec.best_bbox
        flags = rec.best_flags or {}

        self.decisions.append({
            "frame_number": rec.best_frame_number,
            "tracker_id": rec.tracker_id,
            "final_identity_id": rec.final_identity_id,
            "decision_type": decision_type,
            "best_match_id": best_match_id,
            "best_score": best_score,
            "second_match_id": second_match_id,
            "second_score": second_score,
            "threshold": cfg.MATCH_THRESHOLD,
            "margin": cfg.MATCH_MARGIN,
            "crop_path": rec.crop_path,
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
            "quality_score": rec.best_quality_score,
            "best_crop_was_valid": rec.best_crop_was_valid,
            "valid_candidate_count": rec.valid_candidate_count,
            "candidate_count": rec.candidate_count,
            "rejection_reasons": rejection_reasons(flags),
            "is_occluded": flags.get("is_occluded", None),
            "is_blurry": flags.get("is_blurry", None),
            "is_motion_blurred": flags.get("is_motion_blurred", None),
            "is_thin_sliver": flags.get("is_thin_sliver", None),
            "is_full_body": flags.get("is_full_body", None),
            "has_face_detected": rec.has_face,
            "face_weight_used": cfg.FACE_WEIGHT if rec.has_face else 0.0,
        })

        self.mappings.append({
            "tracker_id": rec.tracker_id,
            "final_identity_id": rec.final_identity_id,
            "decision_type": decision_type,
            "match_score": rec.match_score,
            "best_crop_frame": rec.best_frame_number,
            "best_crop_was_valid": rec.best_crop_was_valid,
            "quality_score": rec.best_quality_score,
            "crop_path": rec.crop_path,
        })

    def log_ranking(self, tracker_id, final_identity_id, ranking):
        for rank_idx, (identity_id, score) in enumerate(ranking, start=1):
            self.rankings.append({
                "tracker_id": tracker_id,
                "final_identity_id": final_identity_id,
                "rank": rank_idx,
                "gallery_identity_id": identity_id,
                "similarity_score": score,
            })

    def save(self):
        pd.DataFrame(self.decisions).to_csv(DECISIONS_CSV, index=False)
        pd.DataFrame(self.rankings).to_csv(RANKINGS_CSV, index=False)
        pd.DataFrame(self.mappings).to_csv(MAPPING_CSV, index=False)

# =========================
# ANNOTATOR  (with dual counters + sidebar panel)
# =========================
#
# COUNTER-1  Unique IDs
#   = number of DISTINCT final identity IDs that are currently VISIBLE in this frame.
#   Counts only the resolved IDs whose tracker boxes appear in the current frame.
#   Grows as new people walk in, shrinks when they leave.
#   (e.g. if ID-001, ID-003, ID-005 are on screen right now → counter = 3)
#
# COUNTER-2  Appearances
#   = how many times ANY person ENTERED the frame (cumulative over whole video).
#   Entry = tracker_id appears in this frame but was ABSENT in the previous frame.
#   So a person who walks in, leaves, then walks back in counts as 2.
#   NOT incremented on every frame — only on each fresh entry.
#
# COUNTER-3  Total Unique IDs
#   = total DISTINCT final identity IDs resolved so far in the whole video
#     (cumulative, never shrinks). This is simply len(all_resolved_ids),
#     i.e. total unique ids = total unique count.
#   Unlike COUNTER-1 (which only counts IDs visible RIGHT NOW), this keeps
#   counting every identity that has ever been resolved, even after they
#   leave the frame.
#
# SIDEBAR  (real-time)
#   Shows every identity that has been RESOLVED so far in the video.
#   Similarity score and Top-3 ranking come from the LIVE gallery state
#   (gallery.items), so they reflect the latest OSNet match scores, not
#   a pre-built static snapshot.

class VideoAnnotator:
    SIDEBAR_W  = 280
    THUMB_W    = 64
    THUMB_H    = 120
    ROW_H      = 134
    HEADER_H   = 38
    BG_COLOR   = (30, 30, 30)
    HDR_COLOR  = (50, 50, 50)
    TXT_COLOR  = (230, 230, 230)
    ACC_COLOR  = (0, 200, 100)
    RANK_COLOR = (180, 220, 255)
    FONT       = cv2.FONT_HERSHEY_SIMPLEX

    # --- NEW: resolved-identity boxes are now YELLOW and thicker/bigger,
    # so a correctly Re-ID'd person clearly stands out in the video. ---
    BOX_COLOR_RESOLVED = (0, 255, 255)   # BGR yellow
    BOX_COLOR_PENDING  = (0, 180, 255)   # unchanged orange for "ID pending"
    BOX_THICKNESS      = 4               # was 2
    BOX_LABEL_FONT_SCALE = 0.70          # was 0.55, bigger label text
    BOX_LABEL_THICKNESS  = 2

    def __init__(self):
        # Counter state is based on final ReID labels, not raw tracker IDs.
        self._prev_visible_final_ids = set()
        self._seen_final_ids = set()
        self._appearance_count = 0

    # ------------------------------------------------------------------
    def draw(self, frame, tracks, tracker_to_identity,
             gallery=None, crop_manager=None):
        """
        gallery        : live IdentityGallery object  (for real-time scores/top3)
        crop_manager   : live BestCropManager object  (for thumbnails)
        """
        fh, fw = frame.shape[:2]

        # ── 1. Widen canvas ───────────────────────────────────────────
        canvas = np.zeros((fh, fw + self.SIDEBAR_W, 3), dtype=np.uint8)
        canvas[:, :fw] = frame

        # ── 2. Original bounding-box drawing (unchanged) ──────────────
        for tr in tracks:
            x1, y1, x2, y2 = map(int, tr["bbox"])
            tracker_id = tr["tracker_id"]

            final_info = tracker_to_identity.get(tracker_id)
            if final_info:
                label    = final_info["label"]
                score    = final_info.get("score")
                decision = final_info.get("decision_type")
                if score is not None:
                    label = f"{label} {score:.2f}"
                if decision:
                    label = f"{label} {decision}"
                color = self.BOX_COLOR_RESOLVED
            else:
                if not CFG.DRAW_PENDING_TRACKS:
                    continue
                label = "ID pending"
                color = self.BOX_COLOR_PENDING

            cv2.rectangle(canvas, (x1, y1), (x2, y2), color, self.BOX_THICKNESS)
            y_text = max(26, y1 - 10)
            cv2.rectangle(
                canvas,
                (x1, y_text - 26),
                (x1 + max(150, int(len(label) * 12)), y_text + 6),
                color, -1
            )
            cv2.putText(
                canvas, label,
                (x1 + 4, y_text - 4),
                self.FONT, self.BOX_LABEL_FONT_SCALE, (0, 0, 0),
                self.BOX_LABEL_THICKNESS, cv2.LINE_AA
            )

        # ── 3. COUNTER-1  Unique IDs visible RIGHT NOW ────────────────
        current_tracker_ids = {tr["tracker_id"] for tr in tracks}

        visible_final_ids = set()
        for tid in current_tracker_ids:
            info = tracker_to_identity.get(tid)
            if info:
                visible_final_ids.add(info["label"])

        unique_now = len(visible_final_ids)

        # ── 4. COUNTER-2  Appearance entries (cumulative) ─────────────
        # A new entry = final identity visible NOW but absent in PREVIOUS frame.
        # If the same person leaves and comes back, only this counter increases.
        new_entries = visible_final_ids - self._prev_visible_final_ids
        self._appearance_count += len(new_entries)
        self._prev_visible_final_ids = visible_final_ids

        # ── COUNTER-3  Total Unique IDs (cumulative, whole video so far) ──
        # This grows only when a never-before-seen final identity appears.
        self._seen_final_ids.update(visible_final_ids)
        all_resolved_ids = sorted(self._seen_final_ids)
        total_unique_count = len(all_resolved_ids)

        # ── Draw counter panel (top-left, semi-transparent, bigger + clearer) ──
        ctr_x, ctr_y = 14, 14
        box_w, box_h = 330, 138
        overlay = canvas.copy()
        cv2.rectangle(overlay,
                      (ctr_x, ctr_y),
                      (ctr_x + box_w, ctr_y + box_h),
                      (18, 18, 18), -1)
        cv2.addWeighted(overlay, 0.72, canvas, 0.28, 0, canvas)
        cv2.rectangle(canvas,
                      (ctr_x, ctr_y),
                      (ctr_x + box_w, ctr_y + box_h),
                      (90, 90, 90), 1)

        # Header strip
        cv2.rectangle(canvas,
                      (ctr_x, ctr_y),
                      (ctr_x + box_w, ctr_y + 30),
                      (45, 45, 45), -1)
        cv2.putText(canvas, "LIVE COUNTERS",
                    (ctr_x + 10, ctr_y + 21),
                    self.FONT, 0.55, (255, 255, 255), 2, cv2.LINE_AA)

        rows = [
            ("On screen now",  unique_now,         (0, 255, 140)),
            ("Total appears",  self._appearance_count, (0, 200, 255)),
            ("Total unique IDs", total_unique_count,   (0, 255, 255)),
        ]
        row_y = ctr_y + 30
        row_h = (box_h - 30) // len(rows)
        for i, (row_label, row_value, row_color) in enumerate(rows):
            ry1 = row_y + i * row_h
            ry2 = ry1 + row_h
            # Left accent bar so each counter is instantly distinguishable
            cv2.rectangle(canvas, (ctr_x, ry1 + 3), (ctr_x + 6, ry2 - 3), row_color, -1)
            cv2.putText(canvas, row_label,
                        (ctr_x + 16, ry1 + row_h // 2 + 6),
                        self.FONT, 0.58, (235, 235, 235), 2, cv2.LINE_AA)
            value_text = str(row_value)
            (tw, th), _ = cv2.getTextSize(value_text, self.FONT, 0.85, 2)
            cv2.putText(canvas, value_text,
                        (ctr_x + box_w - tw - 16, ry1 + row_h // 2 + 8),
                        self.FONT, 0.85, row_color, 2, cv2.LINE_AA)
            if i < len(rows) - 1:
                cv2.line(canvas, (ctr_x, ry2), (ctr_x + box_w, ry2), (60, 60, 60), 1)

        # ── 5. Sidebar panel  (real-time from live gallery) ───────────
        sb_x = fw
        canvas[:, sb_x:] = self.BG_COLOR

        # Header
        canvas[:self.HEADER_H, sb_x:] = self.HDR_COLOR
        cv2.putText(canvas, "Identity Panel  [live]",
                    (sb_x + 6, self.HEADER_H - 10),
                    self.FONT, 0.48, self.ACC_COLOR, 1, cv2.LINE_AA)

        # all_resolved_ids already computed above (COUNTER-3) and reused here
        # so the sidebar list and "Total Unique IDs" counter always match.
        y_cursor = self.HEADER_H + 4

        for identity_id in all_resolved_ids:
            if y_cursor + self.ROW_H > fh:
                remaining = len(all_resolved_ids) - all_resolved_ids.index(identity_id)
                cv2.putText(canvas,
                            f"  +{remaining} more…",
                            (sb_x + 6, y_cursor + 18),
                            self.FONT, 0.42, self.TXT_COLOR, 1, cv2.LINE_AA)
                break

            row_y1 = y_cursor
            row_y2 = y_cursor + self.ROW_H

            cv2.line(canvas,
                     (sb_x, row_y1),
                     (sb_x + self.SIDEBAR_W, row_y1),
                     (70, 70, 70), 1)

            # ── Thumbnail ─────────────────────────────────────────────
            thumb_x = sb_x + 4
            thumb_y = row_y1 + 4
            thumb_drawn = False

            if crop_manager is not None:
                for tr_id, info in tracker_to_identity.items():
                    if info["label"] == identity_id:
                        rec = crop_manager.records.get(tr_id)
                        if rec and rec.best_crop is not None:
                            thumb = cv2.resize(
                                rec.best_crop,
                                (self.THUMB_W, self.THUMB_H)
                            )
                            ty2 = min(thumb_y + self.THUMB_H, fh - 1)
                            tx2 = min(thumb_x + self.THUMB_W,
                                      fw + self.SIDEBAR_W - 1)
                            canvas[thumb_y:ty2, thumb_x:tx2] =                                 thumb[:ty2 - thumb_y, :tx2 - thumb_x]
                            cv2.rectangle(canvas,
                                          (thumb_x, thumb_y),
                                          (thumb_x + self.THUMB_W,
                                           thumb_y + self.THUMB_H),
                                          self.ACC_COLOR, 1)
                            thumb_drawn = True
                        break

            if not thumb_drawn:
                cv2.rectangle(canvas,
                              (thumb_x, thumb_y),
                              (thumb_x + self.THUMB_W,
                               thumb_y + self.THUMB_H),
                              (80, 80, 80), 1)
                cv2.putText(canvas, "N/A",
                            (thumb_x + 14, thumb_y + 60),
                            self.FONT, 0.38, (100, 100, 100), 1, cv2.LINE_AA)

            # ── ID label + LIVE similarity + LIVE top-3 ───────────────
            text_x  = sb_x + self.THUMB_W + 10
            line_y  = row_y1 + 16

            # Pull live data from gallery if available
            live_score = None
            live_top3  = []
            if gallery is not None and identity_id in gallery.items:
                item = gallery.items[identity_id]
                live_score = item.get("last_score")

                # Compute live top-3 similarity against ALL other gallery entries
                # using the mean feature of this identity vs every other entry
                all_features = item["features"]
                if all_features:
                    mean_feat = l2_normalize(np.mean(all_features, axis=0))
                    ranking = []
                    for other_id, other_item in gallery.items.items():
                        best = max(
                            float(np.dot(mean_feat, f))
                            for f in other_item["features"]
                        )
                        ranking.append((other_id, best))
                    ranking.sort(key=lambda x: x[1], reverse=True)
                    live_top3 = ranking[:3]

            cv2.putText(canvas, identity_id,
                        (text_x, line_y),
                        self.FONT, 0.52, self.ACC_COLOR, 1, cv2.LINE_AA)

            if live_score is not None:
                cv2.putText(canvas,
                            f"sim: {live_score:.3f}",
                            (text_x, line_y + 17),
                            self.FONT, 0.40, self.TXT_COLOR, 1, cv2.LINE_AA)

            rank_y = line_y + 36
            if live_top3:
                cv2.putText(canvas, "Top-3:",
                            (text_x, rank_y),
                            self.FONT, 0.36, (150, 150, 150), 1, cv2.LINE_AA)
                for ri, (r_id, r_sc) in enumerate(live_top3):
                    ry = rank_y + 14 + ri * 15
                    if ry > row_y2 - 2:
                        break
                    cv2.putText(canvas,
                                f"  {ri+1}. {r_id}  {r_sc:.3f}",
                                (text_x, ry),
                                self.FONT, 0.34, self.RANK_COLOR, 1, cv2.LINE_AA)
            else:
                cv2.putText(canvas, "No data yet",
                            (text_x, rank_y),
                            self.FONT, 0.34, (90, 90, 90), 1, cv2.LINE_AA)

            y_cursor += self.ROW_H

        return canvas

# =========================
# PROCESS ONE TRACK ONLY ONCE
# =========================

def process_track_once(rec, osnet, face_extractor, gallery, matcher, logger, cfg):
    if rec.processed or rec.best_crop is None:
        return

    crop_source = "valid" if rec.best_crop_was_valid else "fallback"
    crop_path = os.path.join(
        CROPS_DIR,
        f"tracker_{rec.tracker_id}_frame_{rec.best_frame_number}_{crop_source}_q_{rec.best_quality_score:.3f}.jpg"
    )
    cv2.imwrite(crop_path, rec.best_crop)
    rec.crop_path = crop_path

    feature = osnet.extract(crop_path)

    # --- NEW: face feature from the SAME best crop (MTCNN detect inside it
    # + FaceNet embed). If no face is found (facing away, too small, etc.)
    # this is simply None and matching falls back to pure OSNet body score
    # for this track, unchanged from before. ---
    face_feature = None
    if cfg.USE_FACE_FEATURE and face_extractor is not None:
        face_feature = face_extractor.extract(rec.best_crop)
    rec.has_face = face_feature is not None

    decision_type, best_id, best_score, second_id, second_score, ranking = matcher.decide(
        feature,
        cfg.MATCH_THRESHOLD,
        cfg.MATCH_MARGIN,
        query_face_feature=face_feature,
        face_weight=cfg.FACE_WEIGHT
    )

    if decision_type == "old":
        final_id = best_id
        gallery.update_identity(final_id, feature, crop_path, rec.best_frame_number, best_score, face_feature=face_feature)
        match_score = best_score
    else:
        final_id = gallery.create_identity(feature, crop_path, rec.best_frame_number, best_score, face_feature=face_feature)
        match_score = best_score

    gallery_img_path = os.path.join(GALLERY_DIR, f"{final_id}_from_tracker_{rec.tracker_id}_{crop_source}.jpg")
    if not os.path.exists(gallery_img_path):
        cv2.imwrite(gallery_img_path, rec.best_crop)

    rec.final_identity_id = final_id
    rec.match_score = match_score
    rec.decision_type = decision_type
    rec.processed = True

    logger.log_ranking(rec.tracker_id, final_id, ranking)
    logger.log_decision(rec, decision_type, best_id, best_score, second_id, second_score, cfg)

# =========================
# MAIN PIPELINE
# =========================

cap = cv2.VideoCapture(CFG.INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {width}x{height}, FPS={fps:.2f}, frames={total_frames}")

cap.release()

yolo = YOLO(CFG.YOLO_MODEL)
quality_evaluator = CropQualityEvaluator(CFG)
crop_manager = BestCropManager(CFG)
osnet = OSNetFeatureExtractor(CFG, device)
face_extractor = FaceFeatureExtractor(CFG, device) if CFG.USE_FACE_FEATURE else None
gallery = IdentityGallery()
matcher = ReIDMatcher(gallery)
logger = DecisionLogger()

frame_tracks = {}

cap = cv2.VideoCapture(CFG.INPUT_VIDEO)
frame_number = 0

print("Pass 1: tracking, crop selection, one-time OSNet ReID...")

with tqdm(total=total_frames) as pbar:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame_number += 1

        results = yolo.track(
            frame,
            persist=True,
            tracker=CFG.TRACKER_CFG,
            classes=[CFG.PERSON_CLASS_ID],
            conf=CFG.DET_CONF,
            verbose=False
        )

        current_tracks = []

        if results and len(results) > 0 and results[0].boxes is not None:
            boxes_obj = results[0].boxes

            if boxes_obj.id is not None:
                xyxy = boxes_obj.xyxy.detach().cpu().numpy()
                ids = boxes_obj.id.detach().cpu().numpy().astype(int)
                confs = boxes_obj.conf.detach().cpu().numpy()

                all_bboxes = [tuple(map(int, b)) for b in xyxy]

                for bbox, tracker_id, det_conf in zip(xyxy, ids, confs):
                    bbox = tuple(map(int, bbox))
                    crop_manager.observe_track(tracker_id, frame_number)

                    valid, quality_score, flags, crop = quality_evaluator.evaluate(
                        frame,
                        bbox,
                        all_bboxes,
                        float(det_conf)
                    )

                    if crop is not None:
                        crop_manager.update_if_better(
                            tracker_id,
                            frame_number,
                            bbox,
                            crop,
                            quality_score,
                            flags,
                            valid
                        )

                    if crop_manager.is_ready(tracker_id, frame_number):
                        rec = crop_manager.records[tracker_id]
                        process_track_once(rec, osnet, face_extractor, gallery, matcher, logger, CFG)

                    current_tracks.append({
                        "tracker_id": int(tracker_id),
                        "bbox": bbox,
                        "conf": float(det_conf)
                    })

        frame_tracks[frame_number] = current_tracks
        pbar.update(1)

cap.release()

print("Finalizing every remaining track that has any usable crop...")

for rec in crop_manager.unprocessed_with_crop():
    if CFG.PROCESS_LOW_QUALITY_AT_END:
        process_track_once(rec, osnet, face_extractor, gallery, matcher, logger, CFG)
    elif rec.best_crop_was_valid and rec.best_quality_score >= CFG.READY_QUALITY_SCORE:
        process_track_once(rec, osnet, face_extractor, gallery, matcher, logger, CFG)

tracker_to_identity = {}
for rec in crop_manager.records.values():
    if rec.processed and rec.final_identity_id is not None:
        tracker_to_identity[rec.tracker_id] = {
            "label": rec.final_identity_id,
            "score": rec.match_score,
            "decision_type": rec.decision_type,
            "crop_source": "valid" if rec.best_crop_was_valid else "fallback",
        }

print("Pass 2: writing final annotated video with stable final IDs...")

# Pass live gallery + crop_manager objects so sidebar scores/top3 are real-time.
# No static top3 snapshot needed — annotator computes from gallery.items each frame.
annotator = VideoAnnotator()
out_width = width + VideoAnnotator.SIDEBAR_W

cap    = cv2.VideoCapture(CFG.INPUT_VIDEO)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (out_width, height))

frame_number = 0

with tqdm(total=total_frames) as pbar:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame_number += 1
        tracks = frame_tracks.get(frame_number, [])

        # Pass live gallery & crop_manager for real-time sidebar data
        canvas = annotator.draw(
            frame, tracks, tracker_to_identity,
            gallery=gallery,
            crop_manager=crop_manager
        )

        writer.write(canvas)
        pbar.update(1)

cap.release()
writer.release()

logger.save()
pd.DataFrame(gallery.rows()).to_csv(GALLERY_CSV, index=False)

if CFG.SAVE_GALLERY_FEATURES_NPY:
    feature_dump = {}
    for identity_id, item in gallery.items.items():
        feature_dump[identity_id] = np.stack(item["features"], axis=0)
    np.save(PATHS["FEATURES_NPY"], feature_dump, allow_pickle=True)

processed_count = sum(1 for rec in crop_manager.records.values() if rec.processed)
fallback_count = sum(1 for rec in crop_manager.records.values() if rec.processed and not rec.best_crop_was_valid)
total_unique_ids = len(gallery.items)  # total unique ids = total unique count

# =========================
# NEW: SUMMARY CSV  (one row per resolved identity, with ALL counts together)
# =========================
# Each row = one final identity_id, with: how many video frames it appeared
# in, first/last seen frame, whether it had a face detected, its crop image
# path, plus the 3 video-overlay counters as constant columns so everything
# the user needs is in one CSV.
summary_rows = []
appearances_per_identity = {}
for tracker_id, info in tracker_to_identity.items():
    appearances_per_identity[info["label"]] = appearances_per_identity.get(info["label"], 0) + 1

for identity_id, item in gallery.items.items():
    summary_rows.append({
        "identity_id": identity_id,
        "num_tracker_ids_merged_into_this_identity": appearances_per_identity.get(identity_id, 0),
        "first_seen_frame": item["first_seen_frame"],
        "last_seen_frame": item["last_seen_frame"],
        "num_osnet_feature_samples": len(item["features"]),
        "had_face_detected": len(item.get("face_features", [])) > 0,
        "representative_crop_image": item["representative_image"],
        "face_weight_used_for_this_run": CFG.FACE_WEIGHT,
        # constant columns -- final totals for the whole video, repeated on every row
        # so the single CSV contains every count the user asked for
        "TOTAL_UNIQUE_IDS_WHOLE_VIDEO": total_unique_ids,
        "TOTAL_APPEARANCE_ENTRIES_WHOLE_VIDEO": annotator._appearance_count,
        "PROCESSED_TRACKER_IDS": processed_count,
        "FALLBACK_CROP_IDS": fallback_count,
    })

pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)

# =========================
# NEW: ZIP of all selected person crop images (one per resolved tracker_id)
# =========================
with zipfile.ZipFile(CROPS_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for rec in crop_manager.records.values():
        if rec.crop_path and os.path.exists(rec.crop_path):
            arcname = os.path.basename(rec.crop_path)
            zf.write(rec.crop_path, arcname=arcname)

print("\nDONE")
print("Annotated video:", OUTPUT_VIDEO)
print("Selected crops:", CROPS_DIR)
print("Gallery images:", GALLERY_DIR)
print("Decisions CSV:", DECISIONS_CSV)
print("Ranking CSV:", RANKINGS_CSV)
print("Mapping CSV:", MAPPING_CSV)
print("Gallery CSV:", GALLERY_CSV)
print("Summary CSV (all counts + ids together):", SUMMARY_CSV)
print("Person crops ZIP:", CROPS_ZIP)
print("Gallery features:", PATHS["FEATURES_NPY"])
print(f"Processed tracker IDs: {processed_count}")
print(f"Fallback-crop IDs used: {fallback_count}")
print(f"Total Unique IDs (total unique ids = total unique count): {total_unique_ids}")
print(f"Face weight used this run (raise CFG.FACE_WEIGHT to use face more): {CFG.FACE_WEIGHT}")

print("\nImportant enforcement:")
print("- Each tracker_id is processed by OSNet only once.")
print("- Valid + CLEAN (non-motion-blurred, non-thin-sliver) crops are preferred first.")
print("- Fallback crops are used only when nothing better appears, and must clear a much")
print("  higher quality floor (0.45) than before (was 0.12, which accepted near-anything).")
print("- Output folder is cleaned for the current video run, so old crops/features are not reused.")
print("- Final IDs come from this run's in-memory OSNet gallery, not from past video embeddings.")
print("- Face feature (MTCNN+FaceNet) is fused at SCORE level with weight FACE_WEIGHT.")
print("  Tracks with no visible/detectable face are completely unaffected (pure OSNet score).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Current-video output folder: /content/reid_botsort_output/current_video_WIN_20260218_14_25_12_Pro
Using custom tracker config: /content/reid_botsort_output/current_video_WIN_20260218_14_25_12_Pro/botsort_custom.yaml
Video: 1920x1080, FPS=29.99, frames=8293
Model: osnet_ain_x1_0
- params: 2,193,616
- flops: 978,878,352
Successfully loaded pretrained weights from "/content/drive/MyDrive/best_model.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Pass 1: tracking, crop selection, one-time OSNet ReID...


 34%|███▍      | 2851/8293 [03:35<08:48, 10.31it/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
files.download(OUTPUT_VIDEO)

In [ ]:
files.download(CROPS_ZIP)
files.download(GALLERY_CSV)
files.download(SUMMARY_CSV)
files.download(CROPS_DIR)
files.download(GALLERY_DIR)
files.download(DECISIONS_CSV)
files.download(RANKINGS_CSV)
files.download(MAPPING_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>